[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/isospectral_drums/isospectral_drums.ipynb)

# Isospectral Drums

This example computes matching standing waves and vibrations on two differently shaped polygonal drums that share identical natural frequencies, answering negatively Kac's question whether one can hear the shape of a drum. Each drum is modelled as a planar membrane assembled from congruent right triangles with its outer rim held fixed. Cutting a vibration of one drum along its triangles, reflecting the pieces, and recombining signed triples produces a valid vibration of the second drum at the same frequency.

Drum vertices live in the plane as intrinsic antivectors constructed by dualizing grid offsets, while membrane displacements live on interior vertices as scalar fields coupled by linear maps. Reading each triangle with `tile_reading` and inserting signed triples back through `tile_reading.adjugate()` builds `transplant = target_assembly(mixing(source_restriction))` without explicit reflection matrices, because vertex ordering on each triangle aligns shared local coordinates across turns and reflections. That single map intertwines the two interior wave operators, `target_laplacian(transplant) == transplant(source_laplacian)`, carrying every standing wave of the source drum onto an eigenmode of the target drum at the same squared frequency. The planar domain pair comes from [Buser, Conway, Doyle, and Semmler](https://arxiv.org/abs/1005.1839), with the matched triangle lattice following [Cleve Moler's transplantation notes](https://blogs.mathworks.com/cleve/2012/08/20/can-one-hear-the-shape-of-a-drum-part-3-transplantation/) and [Gregory Berkolaiko's folding diagrams](https://people.tamu.edu/~gberkolaiko/other/IsoOrigami/folding_drums.html).

In sparse-matrix and piecewise-isometry notation, each destination triangle $\Delta_a$ sums pulled-back fields $u_T|_{\Delta_a}(x) = \sum_{k=1}^3 \sigma_{ak}\, u_S(Q_{ak}^{-1}(x - p_{ak}))$ under orthogonal reflection-rotation matrices $Q_{ak} \in \mathrm{O}(2)$, assembling the interior matrix $T = D_T^{-1} R_T^{\mathsf{T}} \bigl(\sum_{a,k} \sigma_{ak} S_a^{\mathsf{T}} S_{c_{ak}}\bigr) R_S$ so that $L_T T = T L_S$ maps $L_S u_S = \lambda u_S$ to $L_T (T u_S) = \lambda (T u_S)$.

In [ ]:
# Install numga and its examples when running on Google Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA2D
from numga.sparse import Sparse, SparseExtensor
from examples.util.animation import save_figure, show_states
from examples.math.isospectral_drums import render
from examples.math.isospectral_drums.numerics import drum, triangle

ga = VGA2D
context = NumpyContext(ga)
mv = context.multivector
Scalar = ga.gatype.scalar()                                         # Scalar+
# Drum sample positions live in the plane as intrinsic antivectors.
Point = ga.gatype.antivector()                                      # Antivector-

## Two drums from seven triangles

Each drum joins seven congruent right triangles along shared interior edges while holding its outer perimeter fixed. Ordering every triangle's three corners consistently with one reference right triangle aligns shared local coordinates across reflections and quarter-turns.

In [ ]:
n_lattice = 4
source_indices = np.array([
    [0, 1, 4], [5, 1, 4], [5, 9, 4], [5, 9, 6],
    [10, 9, 6], [10, 9, 14], [10, 11, 6],
])
target_indices = np.array([
    [5, 1, 4], [5, 9, 4], [8, 9, 4], [5, 9, 6],
    [10, 9, 6], [10, 11, 6], [10, 11, 14],
])

In [ ]:
def lattice_tiles(tile_indices: np.ndarray, n_lattice: int) -> Point:
    """Row-major grid indices place triangle corners (`[..., 3] Point-`) on the `n_lattice` by `n_lattice` lattice."""
    offsets = np.arange(n_lattice)
    lattice = (mv.x.dual() * offsets[None, :] + mv.y.dual() * offsets[:, None]).reshape((-1,))  # [n_lattice * n_lattice] Point-
    return lattice[tile_indices]


n_tiles = len(source_indices)
source_tiles = lattice_tiles(source_indices, n_lattice)             # [n_tiles, 3] Point-
target_tiles = lattice_tiles(target_indices, n_lattice)             # [n_tiles, 3] Point-
tiles = stack((source_tiles, target_tiles))                         # [2, n_tiles, 3] Point-
save_figure(render.tiles_figure(tiles), "isospectral_drums_tiles");

Matched colours identify the seven numbered right triangles on the source drum at left and the target drum at right, with white interior seams and dark fixed rims.

## Interior wave operators and standing waves

Each interior vertex couples equally to its four horizontal and vertical lattice neighbours while perimeter vertices stay fixed at zero. Reflections across grid axes and tile diagonals map that four-neighbour stencil onto itself, so reflected triangle fields satisfy the same interior wave equation.

In [ ]:
n_subdivisions = 24
n_modes = 9
mixed_modes = np.array([0, 2, 5])
mix_amplitudes = np.array([0.30, 0.22, 0.26])
exposure = 0.45

In [ ]:
def standing_waves(
    laplacian: SparseExtensor, extension: SparseExtensor, n_subdivisions: int, n_modes: int,
) -> tuple[Scalar, Scalar]:
    """Lowest `n_modes` interior standing waves (`[n_modes] Scalar+[V_interior]`) and squared frequencies (`[n_modes] Scalar+`) solve the Dirichlet eigenvalue problem."""
    # Zero-boundary extension preserves the interior inner product.
    measure = extension.adjoint()(extension)                            # (Scalar+ <- Scalar+)[V_interior <- V_interior]
    eigenvalues, modes = laplacian.eigh(measure, n_modes)               # [n_modes] Scalar+, [n_modes] Scalar+[V_interior]
    return eigenvalues, modes * n_subdivisions


source_vertices, source_topology, source_neighbors, source_restriction, source_assembly, source_extension = drum(
    source_indices, n_lattice, n_subdivisions,
)
target_vertices, target_topology, target_neighbors, target_restriction, target_assembly, target_extension = drum(
    target_indices, n_lattice, n_subdivisions,
)
# Missing perimeter neighbours stay at zero displacement.
source_laplacian = (4 * Scalar - source_neighbors) * n_subdivisions**2  # (Scalar+ <- Scalar+)[V_source <- V_source]
target_laplacian = (4 * Scalar - target_neighbors) * n_subdivisions**2  # (Scalar+ <- Scalar+)[V_target <- V_target]
eigenvalues, waves = standing_waves(source_laplacian, source_extension, n_subdivisions, n_modes)

## Signed triangle transplantation

Each destination triangle combines three signed source triangle fields in shared local coordinates. The signs match displacements across interior seams and cancel pairs along the fixed perimeter, while `tile_reading.adjugate()` inserts each summed triangle back into its destination position before seam assembly averages shared interior vertices.

In [ ]:
contributors = np.array([
    [1, 0, 6], [0, 2, 4], [1, 2, 3], [3, 0, 5],
    [4, 1, 5], [5, 2, 6], [4, 3, 6],
])
signs = np.array([
    [1, -1, 1], [1, 1, 1], [1, -1, 1], [1, -1, 1],
    [1, -1, -1], [1, -1, 1], [1, -1, -1],
])
target_tile = 0

In [ ]:
def tile_mixing(contributors: np.ndarray, signs: np.ndarray, n_samples: int) -> SparseExtensor:
    """Signed three-piece sums map source triangle samples onto destination triangle samples (`(Scalar+ <- Scalar+)[n_tiles * n_samples <- n_tiles * n_samples]`)."""
    n_tiles = len(contributors)
    tile_indices = np.arange(n_tiles * n_samples).reshape(n_tiles, n_samples)
    tile_reading = Sparse.selection(tile_indices, n_tiles * n_samples)  # [n_tiles] Scalar+[n_samples <- n_tiles * n_samples]
    # Reading selects one triangle; its adjugate inserts a triangle into that position.
    signed_tiles = tile_reading[contributors] * signs[..., None]        # [n_tiles, 3] Scalar+[n_samples <- n_tiles * n_samples]
    return (tile_reading[:, None].adjugate() * signed_tiles).sum(axis=1).sum(axis=0) * Scalar


def tile_pieces(
    source_field: Scalar, source_restriction: SparseExtensor,
    contributors: np.ndarray, signs: np.ndarray, target_tile: int,
) -> Scalar:
    """Three signed source triangle fields and their sum (`[4] Scalar+[n_samples]`) form destination triangle `target_tile`."""
    n_tiles = len(contributors)
    n_samples = source_restriction.shape[0] // n_tiles
    tile_indices = np.arange(n_tiles * n_samples).reshape(n_tiles, n_samples)
    tile_reading = Sparse.selection(tile_indices, n_tiles * n_samples)  # [n_tiles] Scalar+[n_samples <- n_tiles * n_samples]
    source_tiles = tile_reading * source_restriction(source_field)      # [n_tiles] Scalar+[n_samples]
    contributions = source_tiles[contributors[target_tile]] * signs[target_tile]  # [3] Scalar+[n_samples]
    return concatenate((contributions, contributions.sum(axis=0)[None]), axis=0)


n_samples = source_restriction.shape[0] // n_tiles
mixing = tile_mixing(contributors, signs, n_samples)                    # (Scalar+ <- Scalar+)[n_tiles * n_samples <- n_tiles * n_samples]
transplant = target_assembly(mixing(source_restriction))                # (Scalar+ <- Scalar+)[V_target <- V_source]
pieces = tile_pieces(waves[mixed_modes[-1]] * mix_amplitudes[-1], source_restriction, contributors, signs, target_tile)  # [4] Scalar+[n_samples]
local_vertices, local_topology = triangle(n_subdivisions)
save_figure(render.pieces_figure(local_vertices, local_topology, pieces, exposure), "isospectral_drums_pieces");

The first three panels show the signed source contributions to the first destination triangle, and the fourth panel shows their sum, with orange and blue for opposite displacements and dark curves along zero nodal lines. Opposing contributions cancel along the hypotenuse that lies on the target drum's fixed rim while reinforcing along the two interior legs.

## Vibrations carried across the drums

Superposing three standing waves of different frequencies sets the source drum into a vibration whose nodal pattern shifts continuously. Applying `transplant` at each instant drives a valid vibration of the target drum with the same frequency components, though localized strikes on the two drums would excite those shared modes with different weights.

In [ ]:
n_frames = 72
frame_duration_ms = 40

In [ ]:
def vibration_states(
    waves: Scalar, eigenvalues: Scalar, mixed_modes: np.ndarray, mix_amplitudes: np.ndarray,
    transplant: SparseExtensor, source_extension: SparseExtensor, target_extension: SparseExtensor,
    n_frames: int,
) -> Iterator[tuple[Scalar, Scalar]]:
    """Yields paired full-drum displacement fields (`Scalar+[V_source]`, `Scalar+[V_target]`) over `n_frames` frames of one fundamental period."""
    n_mixed = len(mixed_modes)
    frequencies = eigenvalues[mixed_modes].square_root()                # [n_mixed] Scalar+
    period = 2 * np.pi / frequencies[0]                                 # [] Scalar+
    times = period * np.linspace(0, 1, n_frames, endpoint=False)[:, None]  # [n_frames, 1] Scalar+
    weights = mix_amplitudes * (frequencies * times).cos()              # [n_frames, n_mixed] Scalar+
    for frame_weights in weights:
        source_field = (waves[mixed_modes] * frame_weights).sum(axis=0)  # Scalar+[V_source]
        # Zero-boundary extension restores the fixed rim vertices for display.
        yield source_extension(source_field), target_extension(transplant(source_field))


states = show_states(
    vibration_states(
        waves, eigenvalues, mixed_modes, mix_amplitudes,
        transplant, source_extension, target_extension, n_frames,
    ),
    render.drum_frames,
    (source_vertices, target_vertices),
    (source_topology, target_topology),
    tiles,
    exposure,
    name="isospectral_drums",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

Orange and blue mark upward and downward membrane displacement on the source drum at left and the transplanted target drum at right, while dark curves follow instantaneous nodal lines across the seven outlined triangles.

In [ ]:
# checks
# Every constituent triangle on both drums is a unit-leg right triangle of signed double area +-1.
double_areas = (tiles[:, :, 1] - tiles[:, :, 0]) & (tiles[:, :, 2] - tiles[:, :, 0])  # [2, n_tiles] Scalar-
assert ((double_areas.squared() - 1).abs() < 1e-9).all()

# Transplantation intertwines the two wave operators and preserves seam continuity on all computed standing waves.
commutator = target_laplacian(transplant) - transplant(source_laplacian)  # (Scalar+ <- Scalar+)[V_target <- V_source]
assert (commutator(waves).abs() < 1e-6).all()
assert ((target_restriction(transplant(waves)) - mixing(source_restriction(waves))).abs() < 1e-9).all()

# Streamed vibrations double the squared displacement norm under transplantation and vanish along both rims.
source_norms = stack([pair[0].batch().squared().sum() for pair in states])  # [n_frames] Scalar+
target_norms = stack([pair[1].batch().squared().sum() for pair in states])  # [n_frames] Scalar+
assert ((target_norms - 2 * source_norms).abs() < 1e-9).all()
assert all((pair[0].batch()[source_topology.exterior[0]].abs() < 1e-9).all() for pair in states)
assert all((pair[1].batch()[target_topology.exterior[0]].abs() < 1e-9).all() for pair in states)